# AutoCIM-Agent — 시연 노트북

**CIM(Compute-In-Memory) 반도체 HW-SW 공동설계를 위한 LangGraph 기반 8-노드 폐루프 파이프라인**

이 노트북은 발표 시연용입니다. 아래 순서로 보여줍니다.

| # | 내용 | 예상 시간 |
|---|---|---|
| 1 | 에이전트 그래프 구조 | 즉시 |
| 2 | 1 iteration 실제 실행 (실제 QAT 학습 포함) | 약 12초 |
| 3 | **HITL 인터럽트 → 재개** | 12초 + 즉시 |
| 4 | 실측 결과 (미리 측정해둔 값) | 즉시 |
| 5 | HTML 대시보드 | 즉시 |

> ⚠️ **이 노트북의 학습 규모는 시연용(64장/1에폭)입니다.** 여기서 나오는 정확도(약 0.17)는 아무 의미가 없습니다.
> 실제 측정값은 **4096장 3시드**로 따로 돌린 결과이며 §4에서 보여줍니다.

## 0. 워밍업 — **발표 시작 전에 미리 실행하세요**

이 셀만 미리 돌려두면 이후 모든 셀이 빠르게 실행됩니다 (모델/데이터셋 캐시 로드).

In [1]:
import os
import sys
from pathlib import Path

# 노트북이 notebooks/ 안에 있어도 프로젝트 루트 기준으로 동작하게 맞춤
ROOT = Path.cwd()
if not (ROOT / "graph.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# 시연 규모: 셀 하나가 몇 초 안에 끝나도록. 실제 측정은 4096장으로 별도 수행(§4).
os.environ["AUTOCIM_QAT_TRAIN_SIZE"] = "64"
os.environ["AUTOCIM_QAT_VAL_SIZE"] = "32"
os.environ["AUTOCIM_QAT_TEST_SIZE"] = "64"
os.environ["AUTOCIM_QAT_MAX_EPOCHS"] = "1"

import json
import time

from langgraph.types import Command

from graph import build_graph
from main import build_initial_state
from middleware import register_hw_config
from schemas.config import HWConfig
from nodes.evaluator import MAX_RETRY_LIMIT
from tools.qat import get_qat_backend, get_layer_groups

HW = HWConfig(**json.load(open("examples/hw_configs/default_cim_v1_128x128.json")))
register_hw_config(HW)

t = time.perf_counter()
get_qat_backend("resnet18")          # 모델 + CIFAR10 캐시 워밍업
n_stages = len(get_layer_groups("resnet18"))
print(f"워밍업 완료: {time.perf_counter() - t:.1f}초")
print(f"대상 하드웨어: {HW.crossbar_rows}x{HW.crossbar_cols} 크로스바, "
      f"ADC {HW.adc_bits}bit, DAC {HW.dac_bits}bit, 타일 {HW.num_tiles}개")
print(f"resnet18 탐색 대상 stage: {n_stages}개 → 탐색 공간 {2 * n_stages}차원")

워밍업 완료: 2.7초
대상 하드웨어: 128x128 크로스바, ADC 8bit, DAC 4bit, 타일 16개
resnet18 탐색 대상 stage: 10개 → 탐색 공간 20차원


---
## 1. 에이전트 그래프 구조

`graph.py`가 조립한 **실제 LangGraph 그래프**입니다. 아래 출력은 손으로 그린 게 아니라
컴파일된 그래프에서 직접 추출한 것입니다.

In [2]:
graph = build_graph()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	tuner(tuner)
	mapper(mapper)
	profiler(profiler)
	verifier(verifier)
	evaluator(evaluator)
	precision_verifier(precision_verifier)
	hitl_human_approval(hitl_human_approval)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	evaluator -. &nbsp;Concluded (low confidence)&nbsp; .-> __end__;
	evaluator -. &nbsp;HITL Interrupt&nbsp; .-> hitl_human_approval;
	evaluator -. &nbsp;Re-plan with History&nbsp; .-> planner;
	evaluator -. &nbsp;Converged (Done)&nbsp; .-> precision_verifier;
	hitl_human_approval -. &nbsp;Concluded (low confidence)&nbsp; .-> __end__;
	hitl_human_approval -. &nbsp;Re-plan&nbsp; .-> planner;
	mapper --> verifier;
	planner --> tuner;
	precision_verifier -. &nbsp;Concluded (low confidence)&nbsp; .-> __end__;
	precision_verifier -. &nbsp;HITL Interrupt&nbsp; .-> hitl_human_approval;
	precision_verifier -. &nbsp;Re-plan with History&nbsp; .-> planner;
	

```mermaid
graph TD;
    START([__start__]) --> planner
    planner["@planner - 탐색/대리모델"] --> tuner
    tuner["@tuner - 실제 PyTorch QAT"] --> mapper
    tuner --> profiler
    mapper["@mapper - NoC 매핑"] --> verifier
    profiler["@profiler - 에너지/지연 물리 시뮬"] --> verifier
    verifier["@verifier - IR-drop/노이즈 검증"] --> evaluator
    evaluator["@evaluator - 수렴 판정/Pareto rank"] -- "Re-plan" --> planner
    evaluator -- "HITL Interrupt" --> hitl["hitl_human_approval"]
    evaluator -- "Converged" --> precision["@precision_verifier - 2단계 정밀 재검증"]
    precision -- "Converged" --> END([__end__])
    precision -- "HITL Interrupt" --> hitl
    hitl -- "Re-plan" --> planner
    hitl -- "낮은 신뢰도 종결" --> END
```

**요점**: `@mapper`와 `@profiler`는 서로 독립이라 **병렬 fan-out**으로 실행되고 `@verifier`에서 fan-in 됩니다.
두 노드가 동시에 상태에 쓰기 때문에, 각 노드는 `metrics_store`의 **자기 네임스페이스에만** 쓰도록
강제됩니다(`namespaced_metrics_reducer`) — 그렇지 않으면 경쟁 조건으로 서로의 결과를 덮어씁니다.

---
## 2. 1 iteration 실제 실행

`@planner`가 후보 제안 → `@tuner`가 **실제 PyTorch 그래디언트 하강으로 QAT 학습** →
`@mapper`/`@profiler`가 물리 시뮬레이션 → `@verifier`가 IR-drop/노이즈 검증 → `@evaluator`가 판정.

실행 시간의 대부분은 **로컬 Ollama LLM**이 rationale을 생성하는 시간입니다.

In [3]:
state = build_initial_state("resnet18", HW, target_accuracy=0.99)
cfg = {"configurable": {"thread_id": f"demo-iter-{int(time.time())}"}}

t, done = time.perf_counter(), False
for chunk in graph.stream(state, config=cfg, stream_mode="updates"):
    for node, update in chunk.items():
        if node == "planner":
            d = update["planner_decisions"][0]
            print(f"[@planner  ] {d['search_tag']}  (LLM 사용={d['used_llm']})")
            print(f"             rationale: {str(d['rationale'])[:90]}...")
        elif node == "tuner":
            print(f"[@tuner    ] 실제 QAT 학습 → accuracy="
                  f"{update['metrics_store']['tuner']['data']['accuracy']}")
        elif node == "mapper":
            m = update["metrics_store"]["mapper"]["data"]
            print(f"[@mapper   ] 필요 타일={m['tiles_needed']}개  NoC 지연={m['noc_latency_ms']}ms")
        elif node == "profiler":
            p = update["metrics_store"]["profiler"]["data"]
            print(f"[@profiler ] 에너지={p['energy_pj']:,.0f} pJ")
        elif node == "verifier":
            v = update["metrics_store"]["verifier"]["data"]
            print(f"[@verifier ] 물리검증={v['is_converged']}  IR-drop={v['ir_drop_error_pct']}%  "
                  f"노이즈마진={v['noise_margin_db']}dB")
        elif node == "evaluator":
            print(f"[@evaluator] 수렴={update.get('is_converged')}  (재시도 {update.get('retry_count')})")
            done = True   # 1 iteration(= @evaluator 도달)까지만 보여주고 중단
    if done:
        break

print(f"\n총 {time.perf_counter() - t:.1f}초")
print("※ accuracy가 낮은 건 시연용 64장 학습이기 때문입니다 — 실제 측정은 §4 참고")

[@planner  ] LHS warm-up candidate 1/15  (LLM 사용=True)
             rationale: The per_stage_suggested_candidate (LHS warm-up candidate 1/15) proposes a weight_bits of 2...


[@tuner    ] 실제 QAT 학습 → accuracy=0.0938
[@profiler ] 에너지=832,103 pJ
[@mapper   ] 필요 타일=699개  NoC 지연=0.000368ms
[@verifier ] 물리검증=True  IR-drop=0.064%  노이즈마진=47.16dB
[@evaluator] 수렴=False  (재시도 1)

총 13.8초
※ accuracy가 낮은 건 시연용 64장 학습이기 때문입니다 — 실제 측정은 §4 참고


---
## 3. HITL (Human-in-the-Loop) 인터럽트 — **이 프로젝트의 핵심 경로**

이 파이프라인에서 HITL은 예외 처리가 아니라 **설계상 핵심 경로**입니다.
탐색·근사 모델을 완전히 신뢰할 수 없다는 전제 위에 만들어졌기 때문에,
`__end__`는 "정답 확정"이 아니라 **"사람이 검토할 후보가 준비됨"**을 뜻합니다.

LangGraph v0.2+ 표준인 **동적 `interrupt()`** 만 사용합니다
(`compile()`에 `interrupt_before`를 절대 주지 않음).

아래는 **이미 2번 실패한 세션을 이어받는** 상황입니다 — 체크포인트에서 재개하는 실제 시나리오.

In [4]:
state = build_initial_state("resnet18", HW, target_accuracy=0.99)
state["retry_count"] = MAX_RETRY_LIMIT - 1        # 이미 2회 실패한 세션을 이어받음
cfg = {"configurable": {"thread_id": f"demo-hitl-{int(time.time())}"}}

t = time.perf_counter()
result = graph.invoke(state, config=cfg)
print(f"실행 {time.perf_counter() - t:.1f}초 → 그래프가 멈췄습니다\n")

payload = result["__interrupt__"][0].value
print("=== 사람에게 전달된 인터럽트 페이로드 ===")
for k in ("reason", "iteration_count", "retry_count", "hw_spec_id", "target_accuracy"):
    print(f"  {k:<16}: {payload.get(k)}")

실행 12.6초 → 그래프가 멈췄습니다

=== 사람에게 전달된 인터럽트 페이로드 ===
  reason          : Convergence stalled -- researcher review requested
  iteration_count : 1
  retry_count     : 3
  hw_spec_id      : cim_v1_128x128
  target_accuracy : 0.99


### 재개 — 두 가지 선택지

연구자는 두 가지로 재개할 수 있습니다.

| 선택 | `Command(resume=...)` | 결과 |
|---|---|---|
| 탐색 범위를 좁혀 재시도 | `{"new_bounds": {...}}` | `@planner`로 돌아가 계속 탐색 |
| **지금까지의 최선으로 종결** | `{"conclude_with_best": True}` | 바로 `__end__`, **낮은 신뢰도로 표시** |

아래는 두 번째 경로입니다. `is_converged=True`가 되지만 `low_confidence_conclusion=True`가
함께 붙어서, **"목표를 진짜 만족한 수렴"과 "포기하고 고른 최선"을 절대 혼동하지 않게** 합니다.

In [5]:
t = time.perf_counter()
final = graph.invoke(Command(resume={"conclude_with_best": True}), config=cfg)
print(f"재개 {time.perf_counter() - t:.2f}초\n")
print(f"  is_converged              : {final.get('is_converged')}")
print(f"  low_confidence_conclusion : {final.get('low_confidence_conclusion')}   ← 검토 필요 표시")
print(f"  iteration_count           : {final.get('iteration_count')}")
print(f"  평가한 후보 수            : {len(final.get('candidate_history', []))}")
print("\n→ 이 표시가 붙은 결과는 CLI 최종 출력과 대시보드 모두에서 '검토 필요'로 구분 표시됩니다.")

재개 0.00초

  is_converged              : True
  low_confidence_conclusion : True   ← 검토 필요 표시
  iteration_count           : 1
  평가한 후보 수            : 1

→ 이 표시가 붙은 결과는 CLI 최종 출력과 대시보드 모두에서 '검토 필요'로 구분 표시됩니다.


---
## 4. 실측 결과 — **여기가 진짜 수치입니다**

아래는 시연용 64장이 아니라 **CIFAR10 4096장 · 3시드**로 별도 측정한 결과입니다.

핵심은 "탐색이 최적해를 찾았다"가 아닙니다. 두 가지입니다.

1. **비교 규칙 자체가 틀려 있었습니다.** 탐색은 목표를 넘긴 *첫* 후보에서 멈추고,
   그리드는 전부 평가한 뒤 *최고점*을 보고했습니다. 같은 규칙으로 맞추자
   시드 편차가 사라졌습니다.
2. **실제로 돌려봤더니 물리 모델이 틀려 있었고, 그걸 찾아 고쳤습니다.**


In [6]:
from unicodedata import east_asian_width

# 한글은 모노스페이스에서 두 칸을 차지하는데 f-string 폭 지정은 글자 수로 세므로,
# 그냥 :<20 을 쓰면 한글이 섞인 표의 열이 어긋납니다 (발표 화면에서 특히 눈에 띔).
def pad(text, width):
    shown = sum(2 if east_asian_width(ch) in "WF" else 1 for ch in text)
    return text + " " * max(0, width - shown)

results = json.load(open("baseline_compare_v4_budget_results.json"))
base, search, grid = results["baseline"], results["search"], results["grid"]
b, s = base["summary"], search["summary"]
base_e, search_e = b["energy_pj"]["mean"], s["energy_pj"]["mean"]
bg = grid["best"]
same = next(g for g in grid["results"]
            if g["weight_bits"] == 4 and g["column_pruning_ratio"] == 0.0)

def row(label, conv, acc, energy, note=""):
    print(f'  {pad(label, 34)}{pad(conv, 8)}{acc:>9.3f}{energy:>16,.0f}{note}')

print("=== 같은 정지 규칙으로 맞춘 비교 (resnet18, 4096장, 3시드, 목표 정확도 0.7) ===")
print()
print(f'  {pad("", 34)}{pad("수렴", 8)}{"정확도":>7}{"에너지(pJ)":>13}')
row("베이스라인 (탐색 안 함)", f'{b["n_converged"]}/3', b["accuracy"]["mean"], base_e)
row("탐색 (예산 30회)", f'{s["n_converged"]}/3', s["accuracy"]["mean"], search_e,
    f'   <- 절감 {100*(1-search_e/base_e):.1f}%')
row("그리드 최고점 (w=8, p=0)", f'{len(grid["results"])}/15', bg["accuracy"], bg["energy_pj"])
row("그리드 동일 정확도점 (w=4, p=0)", "-", same["accuracy"], same["energy_pj"])

print()
print("  ※ 정확도는 세 방식이 통계적으로 구분되지 않습니다 (신뢰구간 겹침).")
print("     에너지는 탐색이 베이스라인보다 낮고, 신뢰구간이 겹치지 않습니다.")
print("  ※ 다만 그리드도 동등한 점을 찾았고, 탐색은 30회 / 그리드는 15회를 썼습니다.")
print("     따라서 방어 가능한 주장은 '그리드를 이긴다'가 아니라,")
print("     '미탐색 설계보다 같은 정확도에서 에너지를 덜 쓰고, 전수 그리드의 최선점에 도달한다' 입니다.")

print()
print("=== 정지 규칙을 고치자 시드 편차가 사라졌습니다 ===")
print("  이전 (첫 충족 후보에서 정지) : 0.704 / 0.890 / 0.877   <- 한 시드가 목표를 겨우 넘고 멈춤")
seeds = " / ".join(f'{r["accuracy"]:.3f}' for r in search["results"])
print(f"  이후 (예산 30회 소진)        : {seeds}")
print("  -> 편차의 원인은 탐색 품질이 아니라 정지 규칙이었음이 실측으로 확인됐습니다.")

# Optuna 비교는 아직 예산 규칙으로 재실행하지 않았습니다. 정확도와 수렴 횟수는
# 보정 계수 정정과 무관하므로 그대로 유효하지만, 에너지 pJ는 옛 기준이라 뺐습니다.
try:
    opt = json.load(open("optuna_compare_v2_results.json"))
except FileNotFoundError:
    opt = None
if opt:
    print()
    print("=== 자체 탐색 vs Optuna (동일 파이프라인·동일 정지조건, 3시드) ===")
    for name, arm in opt.items():
        sm = arm["summary"]
        print(f'  {pad(name, 34)}수렴 {sm["n_converged"]}/3   정확도 {sm["accuracy"]["mean"]:.3f}')
    print("  ※ 양쪽 모두 옛 satisficing 정지 규칙으로 측정한 결과입니다 (같은 규칙이므로 비교 자체는 공정).")

=== 같은 정지 규칙으로 맞춘 비교 (resnet18, 4096장, 3시드, 목표 정확도 0.7) ===

                                    수렴        정확도      에너지(pJ)
  베이스라인 (탐색 안 함)           3/3         0.900         920,605
  탐색 (예산 30회)                  3/3         0.900         840,527   <- 절감 8.7%
  그리드 최고점 (w=8, p=0)          15/15       0.905         920,605
  그리드 동일 정확도점 (w=4, p=0)   -           0.900         844,137

  ※ 정확도는 세 방식이 통계적으로 구분되지 않습니다 (신뢰구간 겹침).
     에너지는 탐색이 베이스라인보다 낮고, 신뢰구간이 겹치지 않습니다.
  ※ 다만 그리드도 동등한 점을 찾았고, 탐색은 30회 / 그리드는 15회를 썼습니다.
     따라서 방어 가능한 주장은 '그리드를 이긴다'가 아니라,
     '미탐색 설계보다 같은 정확도에서 에너지를 덜 쓰고, 전수 그리드의 최선점에 도달한다' 입니다.

=== 정지 규칙을 고치자 시드 편차가 사라졌습니다 ===
  이전 (첫 충족 후보에서 정지) : 0.704 / 0.890 / 0.877   <- 한 시드가 목표를 겨우 넘고 멈춤
  이후 (예산 30회 소진)        : 0.899 / 0.899 / 0.902
  -> 편차의 원인은 탐색 품질이 아니라 정지 규칙이었음이 실측으로 확인됐습니다.

=== 자체 탐색 vs Optuna (동일 파이프라인·동일 정지조건, 3시드) ===
  multi_objective (this project)    수렴 1/3   정확도 0.691
  optuna (NSGA-II, direct)          수렴 0/3   정확도 0.524
  ※ 양쪽 모두 옛 satisficing 정

### 실제로 발견한 물리 모델 버그 3건

전부 **실제로 돌려보지 않으면 유닛 테스트로는 잡히지 않는** 종류입니다.

| # | 버그 | 영향 |
|---|---|---|
| 1 | SRAM 크로스바 MAC 에너지가 **정확히 0.0** | NeuroSim C++ 소스까지 추적 — 저항성 셀 모델이 디지털 SRAM에 구조적으로 부적용 |
| 2 | 물리 모델이 **레이어 형상을 아예 안 봄** | resnet18 **69.9배**, mobilenet_v2 **37.3배** 과소평가. 배율이 아키텍처마다 달라 **교차 비교가 왜곡** |
| 3 | `column_pruning_ratio`가 에너지에 **무영향** | 탐색 2개 차원 중 하나가 **보상 없는 순수 손해**였음 |

그리고 고친 뒤 **전부 재측정했고, 기존 결론이 그대로 유지**됐습니다.
"고치니 유리해졌다"가 아니라 **"고쳐도 주장이 살아남았다"** 입니다.

---
## 5. HTML 대시보드

`tools/dashboard.py`가 세션 상태를 그대로 렌더링합니다.
위 §3에서 만든 **낮은 신뢰도 종결** 세션이라, 경고 배지가 함께 표시됩니다.

In [7]:
import html as html_mod

from IPython.display import HTML, display

from tools.dashboard import render_dashboard_html

dashboard = render_dashboard_html(final)
out = Path("notebooks/demo_dashboard.html")
out.write_text(dashboard, encoding="utf-8")
print(f"저장: {out} ({len(dashboard):,} bytes)  ← 브라우저 탭으로 따로 열어도 됩니다")

# srcdoc iframe으로 격리해 표시합니다.
#  - display(HTML(dashboard))로 문서를 그대로 주입하면 대시보드의
#    `body { padding: ... 4vw ... }` 규칙이 Jupyter 페이지 자체의 <body>에
#    적용돼, 확대/축소할 때 레이아웃이 밀리고 오른쪽이 잘립니다.
#  - src="파일경로" 방식은 Jupyter를 어느 디렉터리에서 띄웠는지에 따라
#    URL이 달라져 깨질 수 있어, 내용을 직접 심는 srcdoc을 씁니다.
display(HTML(
    f'<iframe srcdoc="{html_mod.escape(dashboard, quote=True)}" '
    f'style="width:100%; height:760px; border:1px solid #ccc; border-radius:6px;"></iframe>'
))

저장: notebooks\demo_dashboard.html (24,452 bytes)  ← 브라우저 탭으로 따로 열어도 됩니다


C:\Python\AutoCIM-Agent\.venv\Lib\site-packages\IPython\core\display.py:452: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


---
## 정리

- **LangGraph 폐루프 그래프 + 동적 `interrupt()` HITL + 구조화 출력(forced tool-calling)** —
  1단계 교육에서 배운 내용을 실제 연구 문제에 적용
- **LLM은 수치를 정하지 않습니다.** 로컬 모델이 정확도 문제에서 비트 수를 거꾸로 추론하는 걸
  실측으로 발견한 뒤, 역할을 rationale 설명과 이상 징후 플래깅으로 좁혔습니다
- **정직성이 설계 원칙**: 근사/실제 모델 구분 표시, 낮은 신뢰도 종결 표시, 캘리브레이션 출처 배지
- 479개 테스트 · CI · Docker 실제 소자 모델 연동

자세한 내용은 [`README.md`](../README.md), 개념 배경은 [`docs/concepts.md`](../docs/concepts.md).